In [12]:
import plotly.graph_objects as go

In [10]:
from liftlog import empty_log, log_workout, save_log, load_log
import pandas as pd
import matplotlib.pyplot as plt

#test
workout_log = load_log()
print(workout_log.shape)
print(workout_log["date"].dtype) 

(70, 6)
datetime64[us]


In [13]:
# quick inline filter to sanity check the data before we decide on a function shape
exercise_name = "Back Squat"
one_exercise = workout_log[workout_log["exercise_name"] == exercise_name]

print(f"{len(one_exercise)} sets logged for {exercise_name}")
print(f"{one_exercise['date'].nunique()} sessions")
one_exercise.head(8)

43 sets logged for Back Squat
12 sessions


,date,exercise_name,set_number,reps,weight,notes
0,2026-05-24,Back Squat,1,5,165.0,NaN
1,2026-05-24,Back Squat,2,5,165.0,NaN
2,2026-05-24,Back Squat,3,5,165.0,NaN
3,2026-05-24,Back Squat,4,3,170.0,NaN
6,2026-05-28,Back Squat,1,5,170.0,NaN
7,2026-05-28,Back Squat,2,5,170.0,NaN
8,2026-05-28,Back Squat,3,5,170.0,NaN
9,2026-05-28,Back Squat,4,3,175.0,NaN


In [3]:
def get_exercise_history(df, exercise_name):
    """
    Filter the log down to one exercise, sorted by date, then by set_number
    within each day so sets stay in the order they were actually performed.
    """
    filtered = df[df["exercise_name"] == exercise_name].sort_values(["date", "set_number"]) #sort by date and set_number

    if filtered.empty:
        print(f"No data yet for {exercise_name}.")
        return filtered

    session_count = filtered["date"].nunique()
    if session_count == 1:
        print(f"Only one session logged for {exercise_name} — no trend yet.")

    return filtered

In [15]:
squat_history = get_exercise_history(workout_log, "Back Squat")
squat_history["date"].dtype
squat_history.head(8)

,date,exercise_name,set_number,reps,weight,notes
0,2026-05-24,Back Squat,1,5,165.0,NaN
1,2026-05-24,Back Squat,2,5,165.0,NaN
2,2026-05-24,Back Squat,3,5,165.0,NaN
3,2026-05-24,Back Squat,4,3,170.0,NaN
6,2026-05-28,Back Squat,1,5,170.0,NaN
7,2026-05-28,Back Squat,2,5,170.0,NaN
8,2026-05-28,Back Squat,3,5,170.0,NaN
9,2026-05-28,Back Squat,4,3,175.0,NaN


In [19]:
def plot_weight_history(df, title=None):
    """
    Plot weight over evenly spaced set index (row order).
    Expects columns: date, set_number, reps, weight, notes.
    """
    if df.empty:
        print("No data to plot.")
        return None

    # 1-based x so each logged set is one tick apart
    x = list(range(1, len(df) + 1))

    # Format hover fields (handles NaN notes and datetime dates)
    hover_dates = df["date"].dt.strftime("%Y-%m-%d")
    hover_notes = df["notes"].fillna("").astype(str)

    fig = go.Figure(
        go.Scatter(
            x=x,
            y=df["weight"],
            mode="lines+markers",
            marker=dict(size=8),
            customdata=list(zip(
                hover_dates,
                df["set_number"],
                df["reps"],
                df["weight"],
                hover_notes,
            )),
            hovertemplate=(
                "Date: %{customdata[0]}<br>"
                "Set #: %{customdata[1]}<br>"
                "Reps: %{customdata[2]}<br>"
                "Weight: %{customdata[3]} lbs<br>"
                "Notes: %{customdata[4]}<extra></extra>"
            ),
        )
    )

    if title is None and "exercise_name" in df.columns and not df["exercise_name"].empty:
        title = df["exercise_name"].iloc[0]

    fig.update_layout(
        title=title or "Weight history",
        xaxis_title="Set index (chronological order)",
        yaxis_title="Weight (lbs)",
        hovermode="closest",
    )

    return fig

In [20]:
squat_history_chart = plot_weight_history(squat_history)
squat_history_chart

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': [['2026-05-24', 1, 5, 165.0, ''], ['2026-05-24', 2, 5,
                             165.0, ''], ['2026-05-24', 3, 5, 165.0, ''],
                             ['2026-05-24', 4, 3, 170.0, ''], ['2026-05-28', 1, 5,
                             170.0, ''], ['2026-05-28', 2, 5, 170.0, ''],
                             ['2026-05-28', 3, 5, 170.0, ''], ['2026-05-28', 4, 3,
                             175.0, ''], ['2026-06-01', 1, 5, 175.0, ''],
                             ['2026-06-01', 2, 5, 175.0, ''], ['2026-06-01', 3, 5,
                             175.0, ''], ['2026-06-01', 4, 3, 180.0, ''],
                             ['2026-06-04', 1, 5, 175.0, ''], ['2026-06-04', 2, 5,
                             175.0, ''], ['2026-06-04', 3, 5, 175.0, ''],
                             ['2026-06-04', 4, 3, 180.0, 'plateau week'],
                             ['2026-06-08', 1, 5, 180.0, ''], ['2026-06-08', 2, 5,
                             180.0, ''], ['2026-06-08', 3, 5, 180.0, ''],
                             ['2026-06-08', 4, 3, 185.0, ''], ['2026-06-11', 1, 5,
                             180.0, ''], ['2026-06-11', 2, 5, 180.0, ''],
                             ['2026-06-11', 3, 5, 180.0, ''], ['2026-06-11', 4, 3,
                             185.0, 'plateau week'], ['2026-06-15', 1, 5, 182.5,
                             ''], ['2026-06-15', 2, 5, 182.5, ''], ['2026-06-15',
                             3, 3, 187.5, ''], ['2026-06-21', 1, 5, 185.0, 'felt
                             strong'], ['2026-06-21', 2, 5, 185.0, ''],
                             ['2026-06-21', 3, 3, 190.0, 'grinder'], ['2026-06-25',
                             1, 5, 190.0, ''], ['2026-06-25', 2, 5, 190.0, ''],
                             ['2026-06-25', 3, 5, 190.0, ''], ['2026-06-25', 4, 3,
                             195.0, 'new PR'], ['2026-06-26', 1, 5, 185.0, 'felt
                             strong'], ['2026-06-26', 2, 5, 185.0, ''],
                             ['2026-06-26', 3, 3, 190.0, 'grinder'], ['2026-06-27',
                             1, 5, 185.0, 'felt strong'], ['2026-06-27', 2, 5,
                             195.0, ''], ['2026-06-27', 3, 2, 205.0, 'very hard'],
                             ['2026-07-07', 1, 5, 185.0, 'felt strong'],
                             ['2026-07-07', 2, 5, 195.0, ''], ['2026-07-07', 3, 2,
                             205.0, 'very hard']],
              'hovertemplate': ('Date: %{customdata[0]}<br>Set ' ... '{customdata[4]}<extra></extra>'),
              'marker': {'size': 8},
              'mode': 'lines+markers',
              'type': 'scatter',
              'x': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
                    19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
                    35, 36, 37, 38, 39, 40, 41, 42, 43],
              'y': {'bdata': ('AAAAAACgZEAAAAAAAKBkQAAAAAAAoG' ... 'AAACBnQAAAAAAAYGhAAAAAAACgaUA='),
                    'dtype': 'f8'}}],
    'layout': {'hovermode': 'closest',
               'template': '...',
               'title': {'text': 'Back Squat'},
               'xaxis': {'title': {'text': 'Set index (chronological order)'}},
               'yaxis': {'title': {'text': 'Weight (lbs)'}}}
})